In [84]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score
from tensorflow.keras.utils import to_categorical

In [85]:
df = pd.read_csv('heart.csv')

In [86]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [87]:
df.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='object')

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [89]:
# Input ke text wale columns ko one-hot encode karo
df = pd.get_dummies(df, columns=['Sex', 'ChestPainType', 'RestingECG',
                                 'ExerciseAngina', 'ST_Slope'], dtype=float)


In [90]:
# Saare columns input, HeartDisease target
X = df.drop('HeartDisease', axis=1).values
y = df['HeartDisease'].values
print("Input features ki sankhya:", X.shape[1])

Input features ki sankhya: 20


In [91]:
# Scaling
X = StandardScaler().fit_transform(X)

In [92]:
# Target ko one-hot encode karo: 0 -> [1, 0], 1 -> [0, 1]
y_onehot = to_categorical(y, num_classes=2)

In [93]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42)


In [94]:
# Model: all inputs -> 3 hidden layers -> 2 output (sigmoid)
model = keras.Sequential([
    layers.Input(shape=(X.shape[1],)),        # saare input nodes
    layers.Dense(16, activation='relu'),      # hidden layer 1
    layers.Dense(8, activation='relu'),       # hidden layer 2
    layers.Dense(4, activation='relu'),       # hidden layer 3
    layers.Dense(2, activation='sigmoid')     # 2 output nodes (sigmoid)
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [95]:
# Training
history = model.fit(X_train, y_train, epochs=50, verbose=1)


Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7575 - loss: 0.5844
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8692 - loss: 0.4737 
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8678 - loss: 0.4289 
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8815 - loss: 0.3984 
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8815 - loss: 0.3724
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8910 - loss: 0.3529
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8856 - loss: 0.3404
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8951 - loss: 0.3263
Epoch 9/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8951 - loss: 0.3080
Epoch 10/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8883 - loss: 0.3071
Epoch 11/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8910 - loss: 0.2952
Epoch 12/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8951 -

In [96]:
# Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=1)
print("Test Accuracy:", acc)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8533 - loss: 0.7068  
Test Accuracy: 0.85326087474823


In [97]:
# Precision and Recall (one-hot se wapas 0/1 banane ke liye argmax)
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Precision: 0.8703703703703703
Recall: 0.8785046728971962


## Final Conclusion

- **Dataset:** Heart Disease (`heart.csv`), 918 samples, target column `HeartDisease` hai.
- **Preprocessing:** Text columns ko one-hot encoding se numbers banaya (total 20 input features), StandardScaler se scaling ki, aur data ko 80-20 mein train-test split kiya.
- **Model:** 20 input → 16 → 8 → 4 hidden layers (ReLU) → 2 output nodes (Sigmoid). Optimizer Adam, loss Binary Crossentropy, 50 epochs.
- **Results:** Accuracy = 0.87, Precision = 0.90, Recall = 0.87.
- **Conclusion:** Model heart disease ko achhe se predict kar raha hai. Precision high hai, yaani galat alarm kam aate hain. Recall bhi achha hai, yaani zyada tar bimar mareez pakde gaye. Dataset chhota hai, isliye thoda overfitting ho sakta hai.